# Huấn luyện YOLOv8 trên Google Colab (Phiên bản Đầy đủ Metric)

Notebook này bổ sung các cell trực quan hóa để bạn đánh giá mô hình một cách toàn diện sau khi train.

## Bước 1: Cài đặt thư viện

In [ ]:
!pip install -q ultralytics
import ultralytics
from IPython.display import Image, display
import os
ultralytics.checks()

## Bước 2: Giải nén Dataset

In [ ]:
zip_path = '/content/datasets.zip'
if not os.path.exists(zip_path):
    print(" Không tìm thấy datasets.zip!")
else:
    !unzip -q {zip_path} -d /content/
    print(" Giải nén thành công.")

!ls -d /content/images

## Bước 3: Tạo File Cấu Hình và Script Huấn Luyện

In [ ]:
# 1. Tạo file train.py
train_py_content = """
import argparse
from ultralytics import YOLO

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data', default='leanbot_data.yaml')
    parser.add_argument('--epochs', type=int, default=50)
    parser.add_argument('--batch', type=int, default=16)
    parser.add_argument('--name', default='leanbot_colab')
    return parser.parse_args()

if __name__ == '__main__':
    args = parse_args()
    model = YOLO('yolov8n.pt')
    model.train(
        data=args.data,
        epochs=args.epochs,
        batch=args.batch,
        device=0,
        name=args.name,
        degrees=10.0, fliplr=0.5, flipud=0.1
    )
"""
with open('train.py', 'w') as f:
    f.write(train_py_content.strip())

# 2. Tạo file leanbot_data.yaml
yaml_content = """
path: /content
train: images
val: images

names:
  0: Leanbot
"""
with open('leanbot_data.yaml', 'w') as f:
    f.write(yaml_content.strip())
print(" Đã cấu hình xong.")

## Bước 4: Bắt đầu Huấn Luyện

In [ ]:
!python train.py --epochs 50 --batch 16

## Bước 5: Đánh giá mô hình bằng Hình ảnh và Biểu đồ


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

result_dir = Path('/content/runs/detect/leanbot_colab')

def show_image(filenames, title):
    # filenames có thể là một chuỗi hoặc một list các tên file dự phòng
    if isinstance(filenames, str): filenames = [filenames]
    
    found = False
    for fname in filenames:
        img_path = result_dir / fname
        if os.path.exists(img_path):
            print(f"\n--- {title} ({fname}) ---")
            display(Image(filename=str(img_path), width=800))
            found = True
            break
    if not found:
        print(f"Không tìm thấy biểu đồ cho: {title} (Thử tìm: {filenames})")

# Hiển thị các metric chính
show_image('results.png', "Biểu đồ Kết quả Huấn luyện")
show_image(['confusion_matrix_normalized.png', 'confusion_matrix.png'], "Ma trận nhầm lẫn")

# Hiển thị ảnh so sánh
show_image('val_batch0_labels.jpg', "Ảnh gốc có Label")
show_image('val_batch0_pred.jpg', "Ảnh mô hình tự dự đoán")

# Hiển thị đường cong (Xử lý cả tên file cũ và mới BoxF1/BoxPR)
show_image(['F1_curve.png', 'BoxF1_curve.png'], "Đường cong F1-Score")
show_image(['PR_curve.png', 'BoxPR_curve.png'], "Đường cong Precision-Recall")

## Bước 6: Chạy thử Inference (Dự đoán) trên ảnh mẫu


In [ ]:
from ultralytics import YOLO
import glob
import cv2
from google.colab.patches import cv2_imshow

model_path = '/content/runs/detect/leanbot_colab/weights/best.pt'
if os.path.exists(model_path):
    model = YOLO(model_path)
    sample_images = glob.glob('/content/images/*.jpg')[:3]

    if sample_images:
        results = model.predict(source=sample_images, conf=0.25)
        for result in results:
            res_plotted = result.plot()
            cv2_imshow(res_plotted)
            print(f" Kết quả dự đoán cho: {result.path}")
    else:
        print("Thư mục /content/images trống!")
else:
    print(" Không tìm thấy model best.pt")

## Bước 7: Tải về Model Tốt Nhất

In [ ]:
from google.colab import files
best_model = '/content/runs/detect/leanbot_colab/weights/best.pt'
if os.path.exists(best_model):
    files.download(best_model)
else:
    print(" Không tìm thấy best.pt")